# Notebook 14 — Trace + Eval Driven Improvement

This notebook is meant to be **reused after every meaningful Research Deep Agent change**.

Use it when you:
- change prompts
- add or modify tools
- add memory / Store behavior
- add Skills
- add middleware
- add subagents
- change backend/checkpointer behavior
- change models
- change routing/delegation
- optimize latency, token use, or search behavior

The core loop:

```text
Run agent
   ↓
Inspect trace
   ↓
Classify failure
   ↓
Form hypothesis
   ↓
Design targeted eval
   ↓
Make ONE change
   ↓
Rerun same suite
   ↓
Compare quality + trajectory + economics
   ↓
Keep / revert
```

The goal is to replace random prompt tweaking with an experimental discipline.

## Structure

```text
14.1  Why traces and evals belong together
14.2  Trace anatomy
14.3  Failure taxonomy
14.4  Trace symptom → eval hypothesis
14.5  Compact reusable dataset
14.6  Stable async runner
14.7  Deterministic checks
14.8  Rubric evaluation
14.9  Trajectory evaluation
14.10 Baseline vs candidate
14.11 Diagnose memory / skills / middleware / subagents
14.12 Avoid eval overfitting
14.13 Improvement loop
14.14 Economics bridge
```

Design principle:

> Keep the **runner plumbing stable** and swap only the agent variant under test.

# 14.1 — Why traces and evals belong together

A trace answers:

```text
What happened?
```

An eval answers:

```text
Was it good?
```

You need both.

Example:

```text
Trace:
7 web searches

Eval:
quality = 4.2 / 5
```

That still leaves:

> Were those seven searches necessary?

So the useful unit of analysis is:

```text
trajectory
+
quality
+
cost / latency
```

# 14.2 — Trace anatomy for the research agent

Inspect:

```text
user query
   ↓
model call
   ↓
memory retrieved?
   ↓
skill selected?
   ↓
subagent delegated?
   ↓
web/tool calls
   ↓
summarization?
   ↓
retries?
   ↓
final answer
```

Useful trace questions:
- How many model calls?
- Which tools?
- Were calls repeated?
- Which skill?
- Which memories?
- Which subagent?
- Was delegation necessary?
- Did summarization preserve important information?
- Did the agent repeat work?

## 14.2A — Reuse production tracing

Prefer your existing package tracer and App Insights/Foundry traces.

The notebook should **coordinate and analyze experiments**, not create a second observability stack.

In [ ]:
import inspect
import deep_agents_foundry as daf

print(sorted(name for name in dir(daf) if not name.startswith("_")))

In [ ]:
try:
    from deep_agents_foundry.telemetry import (
        build_azure_tracer,
        attach_tracing,
        build_traced_research_agent,
    )
    print("Tracing helpers loaded")
    print("build_traced_research_agent:", inspect.signature(build_traced_research_agent))
except Exception as exc:
    print("Tracing helper issue:", exc)

try:
    from deep_agents_foundry.evaluation import (
        deterministic_evaluation,
        extract_trajectory_features,
        aggregate_usage,
        trajectory_summary,
        evaluate_with_rubric,
        RESEARCH_RUBRIC,
    )
    print("Evaluation helpers loaded")
    print("evaluate_with_rubric:", inspect.signature(evaluate_with_rubric))
except Exception as exc:
    print("Evaluation helper issue:", exc)

# 14.3 — Failure taxonomy

Use a stable taxonomy so every bad run does not become a vague “prompt problem.”

```text
A. Retrieval failure
B. Reasoning failure
C. Tool-use failure
D. Context failure
E. Memory failure
F. Skill failure
G. Subagent failure
H. Synthesis failure
I. Economics failure
```

Typical examples:
- bad search query
- evidence found but interpreted poorly
- wrong tool / repeated tool
- irrelevant context / lost summary detail
- stale memory
- wrong skill
- unnecessary delegation
- weak final synthesis
- too much work for too little gain

In [ ]:
FAILURE_TAXONOMY = {
    "retrieval": ["bad_search_query", "poor_source_quality", "insufficient_evidence"],
    "reasoning": ["misinterpreted_evidence", "weak_tradeoff_reasoning"],
    "tool_use": ["wrong_tool", "unnecessary_tool", "repeated_tool_call"],
    "context": ["missing_context", "irrelevant_context", "summary_lost_detail"],
    "memory": ["wrong_memory", "stale_memory", "should_not_have_been_remembered"],
    "skill": ["wrong_skill", "skill_not_selected", "skill_not_followed"],
    "subagent": ["unnecessary_delegation", "wrong_specialist", "duplicate_specialist_work"],
    "synthesis": ["weak_final_synthesis", "unsupported_conclusion"],
    "economics": ["too_much_work_for_gain", "latency_without_value"],
}

# 14.4 — From trace symptom → eval hypothesis

Best habit:

> Every code change should correspond to an observed failure and a measurable hypothesis.

Example:

```text
Trace symptom:
3 nearly identical searches

Hypothesis:
agent is over-searching

Change:
tighten search/tool policy

Eval:
did search count fall WITHOUT reducing answer quality?
```

Another:

```text
Trace:
technology skill loaded

Output:
trade-off analysis still weak

Hypothesis:
skill procedure is weak or not followed

Eval:
trade-off coverage rubric
```

In [ ]:
from dataclasses import dataclass

@dataclass
class ImprovementHypothesis:
    observed_symptom: str
    failure_category: str
    hypothesis: str
    proposed_change: str
    primary_metric: str
    guardrail_metric: str

example_hypothesis = ImprovementHypothesis(
    observed_symptom="Three near-duplicate searches",
    failure_category="tool_use",
    hypothesis="Agent is over-searching",
    proposed_change="Tighten search policy",
    primary_metric="web_searches",
    guardrail_metric="quality_score",
)

example_hypothesis

# 14.5 — Compact reusable evaluation dataset

Start with 10–20 representative prompts, not hundreds.

Cover:
- simple
- medium
- complex
- tool-routing
- memory
- skill routing
- subagent delegation
- long-context behavior
- negative/simple cases where the agent should **not** over-agent

In [ ]:
EVAL_CASES = [
    {
        "id": "simple_definition",
        "complexity": "simple",
        "prompt": "What are Microsoft Foundry Hosted Agents?",
        "expected": {"needs_web": True, "max_searches": 2, "should_delegate": False},
    },
    {
        "id": "medium_architecture",
        "complexity": "medium",
        "prompt": (
            "Explain Microsoft Foundry Hosted Agents architecture, "
            "state/session boundaries, and identity model."
        ),
        "expected": {"needs_web": True, "max_searches": 4},
    },
    {
        "id": "complex_comparison",
        "complexity": "complex",
        "prompt": (
            "Compare Microsoft Foundry Hosted Agents with a custom AKS-hosted "
            "LangGraph agent for enterprise workloads. Cover architecture, identity, "
            "state ownership, scaling, observability, reliability, lock-in, and economics."
        ),
        "expected": {"needs_web": True, "max_searches": 8},
    },
    {
        "id": "tool_negative",
        "complexity": "simple",
        "prompt": "Explain the difference between a tool and middleware in one paragraph.",
        "expected": {"needs_web": False, "max_searches": 0, "should_delegate": False},
    },
    {
        "id": "skill_routing",
        "complexity": "medium",
        "prompt": "Research LangGraph architecture and explain its most important design trade-offs.",
        "expected": {"expected_skill": "technology-research"},
    },
    {
        "id": "subagent_identity",
        "complexity": "complex",
        "prompt": (
            "Research Hosted Agent authentication, managed identity, RBAC, "
            "credential flow, and important security boundaries."
        ),
        "expected": {"expected_subagent": "identity-researcher"},
    },
]

## 14.5A — Separate DEV and held-out cases

```text
DEV
→ inspect frequently
→ diagnose
→ iterate

HELD-OUT
→ run less often
→ confirm generalization
```

This reduces eval overfitting.

In [ ]:
DEV_CASE_IDS = {
    "simple_definition",
    "medium_architecture",
    "tool_negative",
    "skill_routing",
}

HELD_OUT_CASE_IDS = {
    "complex_comparison",
    "subagent_identity",
}

dev_cases = [c for c in EVAL_CASES if c["id"] in DEV_CASE_IDS]
held_out_cases = [c for c in EVAL_CASES if c["id"] in HELD_OUT_CASE_IDS]

print("dev:", len(dev_cases))
print("held out:", len(held_out_cases))

# 14.6 — Stable reusable async runner

This is the most important code in the notebook.

The runner stays stable while the agent changes.

That means prompt changes, memory, Skills, middleware, subagents, or model changes
can all use the same harness.

In [ ]:
import time
from typing import Any

from deep_agents_foundry import content_text, thread_config


def extract_final_text(result: dict) -> str:
    messages = result.get("messages", [])
    if not messages:
        return ""
    return content_text(messages[-1])


async def run_case(
    agent,
    case: dict,
    *,
    run_label: str,
    thread_prefix: str = "eval",
    context: Any = None,
) -> dict:
    thread_id = f"{thread_prefix}-{run_label}-{case['id']}"
    config = thread_config(thread_id)

    payload = {
        "messages": [
            {
                "role": "user",
                "content": case["prompt"],
            }
        ]
    }

    started = time.perf_counter()

    if context is None:
        result = await agent.ainvoke(payload, config=config)
    else:
        result = await agent.ainvoke(payload, config=config, context=context)

    elapsed = time.perf_counter() - started

    return {
        "case_id": case["id"],
        "complexity": case["complexity"],
        "prompt": case["prompt"],
        "expected": case.get("expected", {}),
        "run_label": run_label,
        "latency_seconds": elapsed,
        "result": result,
        "final_text": extract_final_text(result),
    }

In [ ]:
async def run_suite(
    agent,
    cases,
    *,
    run_label: str,
    context=None,
):
    records = []

    for case in cases:
        record = await run_case(
            agent,
            case,
            run_label=run_label,
            context=context,
        )
        records.append(record)
        print(
            f"{run_label:20s} | "
            f"{case['id']:24s} | "
            f"{record['latency_seconds']:.2f}s"
        )

    return records

### Why the runner should be boring

That is a feature.

```text
stable runner
+
stable dataset
+
changing agent
```

gives interpretable experiments.

# 14.7 — One adapter cell: build baseline and candidate

This should be the main cell you edit after a feature change.

Examples:

```text
baseline
→ known-good current agent

candidate
→ new prompt / memory / skill / middleware / subagent version
```

In [ ]:
from deep_agents_foundry import build_research_agent

baseline_agent = build_research_agent()

# Replace only this builder when testing a new capability.
candidate_agent = build_research_agent()

print("Baseline:", type(baseline_agent).__name__)
print("Candidate:", type(candidate_agent).__name__)

## 14.7A — Traced variants

If you want every eval run in Foundry/App Insights, use your production tracing wrapper:

```python
baseline_agent = build_traced_research_agent(...)
candidate_agent = build_traced_research_agent(...)
```

Keep tracing orthogonal to the runner.

# 14.8 — Deterministic checks

Run cheap checks before an LLM judge.

Examples:
- answer non-empty
- citations present
- search/tool budget respected
- no delegation for simple case
- expected skill/subagent behavior
- no obvious repeated calls

These are cheap, stable, and easy to debug.

In [ ]:
def local_deterministic_checks(record: dict) -> dict:
    text = record["final_text"]

    checks = {
        "non_empty_answer": bool(text.strip()),
        "has_citation_like_content": (
            "http://" in text
            or "https://" in text
            or "【" in text
            or "[http" in text
        ),
    }

    try:
        checks["production_deterministic"] = deterministic_evaluation(record["result"])
    except Exception as exc:
        checks["production_deterministic_error"] = str(exc)

    return checks

# 14.9 — Trajectory extraction

Prefer production helpers:

```text
extract_trajectory_features(...)
aggregate_usage(...)
trajectory_summary(...)
```

The final answer tells you **what came out**.

The trajectory tells you **how much work and what behavior produced it**.

In [ ]:
def attach_trajectory_features(record: dict) -> dict:
    out = dict(record)

    try:
        out["trajectory_features"] = extract_trajectory_features(record["result"])
    except Exception as exc:
        out["trajectory_features"] = {}
        out["trajectory_feature_error"] = str(exc)

    try:
        out["usage"] = aggregate_usage(record["result"])
    except Exception as exc:
        out["usage"] = {}
        out["usage_error"] = str(exc)

    try:
        out["trajectory_text"] = trajectory_summary(record["result"])
    except Exception as exc:
        out["trajectory_text"] = ""
        out["trajectory_summary_error"] = str(exc)

    return out

## 14.9A — What trajectory evaluation should catch

Examples:

```text
simple question
→ should not invoke subagent

technical research
→ relevant skill may load

identity-heavy task
→ identity specialist may be appropriate

web research
→ searches should add new evidence

long context
→ compaction should preserve key facts
```

# 14.10 — Rubric evaluation

Use an LLM judge for qualities that are hard to encode deterministically:

```text
correctness
completeness
source quality
architecture clarity
trade-off depth
decision usefulness
```

Best practices:
- keep rubric stable
- keep judge model explicit
- blind architecture labels where possible
- use deterministic checks as guardrails

In [ ]:
from deep_agents_foundry.model import build_model

judge_model = build_model()
print(type(judge_model).__name__)

## Reusable rubric adapter

Because evaluator signatures may evolve, keep the adaptation in **one cell**.

If `evaluate_with_rubric` changes, update only this wrapper.

In [ ]:
async def rubric_score_record(record: dict, judge_model) -> dict:
    try:
        value = evaluate_with_rubric(
            judge_model=judge_model,
            user_request=record["prompt"],
            response=record["final_text"],
            rubric=RESEARCH_RUBRIC,
        )

        if inspect.isawaitable(value):
            value = await value

        return {
            "rubric_result": value,
            "rubric_error": None,
        }

    except TypeError as exc:
        return {
            "rubric_result": None,
            "rubric_error": (
                "Evaluator signature differs. Inspect evaluate_with_rubric and "
                f"update this adapter only. Details: {exc}"
            ),
        }

    except Exception as exc:
        return {
            "rubric_result": None,
            "rubric_error": str(exc),
        }

# 14.11 — Trajectory rubric

Final-answer quality and trajectory quality are different.

A strong answer can still be produced with:
- duplicate searches
- unnecessary subagents
- excessive retries
- irrelevant memory
- needless skill loading

Trajectory dimensions:

```text
1. Tool necessity
2. Tool selection
3. Search novelty
4. Delegation appropriateness
5. Context discipline
6. Retry discipline
7. Work efficiency
```

Do not reward “fewer steps” blindly.

Reward **necessary, productive steps**.

In [ ]:
TRAJECTORY_RUBRIC = {
    "tool_necessity": "Were tools used only when needed?",
    "tool_selection": "Were the correct tools chosen?",
    "search_novelty": "Did searches add new evidence rather than duplicate?",
    "delegation": "Was subagent delegation appropriate?",
    "context_discipline": "Was irrelevant context avoided?",
    "retry_discipline": "Were retries justified?",
    "work_efficiency": "Did agentic work contribute to the result?",
}

# 14.12 — Reusable end-to-end evaluation function

This is the function you should rerun after future capability changes.

In [ ]:
async def evaluate_agent_variant(
    agent,
    cases,
    *,
    run_label,
    judge_model=None,
    context=None,
):
    records = await run_suite(
        agent,
        cases,
        run_label=run_label,
        context=context,
    )

    enriched = []

    for record in records:
        record = dict(record)
        record["deterministic"] = local_deterministic_checks(record)
        record = attach_trajectory_features(record)

        if judge_model is not None:
            record["rubric"] = await rubric_score_record(record, judge_model)

        enriched.append(record)

    return enriched

In [ ]:
async def compare_variants(
    baseline_agent,
    candidate_agent,
    cases,
    *,
    judge_model=None,
    baseline_label="baseline",
    candidate_label="candidate",
    context=None,
):
    baseline_records = await evaluate_agent_variant(
        baseline_agent,
        cases,
        run_label=baseline_label,
        judge_model=judge_model,
        context=context,
    )

    candidate_records = await evaluate_agent_variant(
        candidate_agent,
        cases,
        run_label=candidate_label,
        judge_model=judge_model,
        context=context,
    )

    return baseline_records, candidate_records

## Example use

After configuring baseline and candidate:

```python
baseline_records, candidate_records = await compare_variants(
    baseline_agent,
    candidate_agent,
    dev_cases,
    judge_model=judge_model,
)
```

Only after the candidate looks better on DEV:

```python
... run held_out_cases ...
```

In [ ]:
# Uncomment when ready for real evaluation calls:
#
# baseline_records, candidate_records = await compare_variants(
#     baseline_agent,
#     candidate_agent,
#     dev_cases,
#     judge_model=judge_model,
# )

# 14.13 — Flatten records into a dataframe

Keep both:
- raw records for diagnosis
- flattened dataframe for comparisons

In [ ]:
import pandas as pd


def numeric_from_mapping(mapping, keys):
    if not isinstance(mapping, dict):
        return None
    for key in keys:
        value = mapping.get(key)
        if isinstance(value, (int, float)):
            return value
    return None


def records_to_dataframe(records):
    rows = []

    for r in records:
        usage = r.get("usage", {}) or {}
        traj = r.get("trajectory_features", {}) or {}
        deterministic = r.get("deterministic", {}) or {}

        rows.append({
            "case_id": r["case_id"],
            "complexity": r["complexity"],
            "run_label": r["run_label"],
            "latency_seconds": r["latency_seconds"],
            "final_text": r["final_text"],
            "non_empty_answer": deterministic.get("non_empty_answer"),
            "has_citation_like_content": deterministic.get("has_citation_like_content"),
            "input_tokens": numeric_from_mapping(
                usage, ["input_tokens", "prompt_tokens", "input"]
            ),
            "output_tokens": numeric_from_mapping(
                usage, ["output_tokens", "completion_tokens", "output"]
            ),
            "model_calls": numeric_from_mapping(
                traj, ["model_calls", "llm_calls"]
            ),
            "tool_calls": numeric_from_mapping(
                traj, ["tool_calls"]
            ),
            "web_searches": numeric_from_mapping(
                traj, ["web_searches", "searches"]
            ),
            "subagent_calls": numeric_from_mapping(
                traj, ["subagent_calls", "task_calls"]
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
def compare_summary(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [
        "latency_seconds",
        "input_tokens",
        "output_tokens",
        "model_calls",
        "tool_calls",
        "web_searches",
        "subagent_calls",
    ]

    available = [c for c in numeric_cols if c in df.columns]

    return (
        df.groupby(["run_label", "complexity"], dropna=False)[available]
        .mean(numeric_only=True)
        .reset_index()
    )

# 14.14 — Diagnose each subsystem explicitly

## Memory
Ask:
- Was the right memory retrieved?
- Was it still true?
- Should it have been stored?
- Did it improve the task?

## Skills
Ask:
- Was the right skill selected?
- Was it necessary?
- Did the procedure improve output?

## Middleware
Ask:
- Did summarization preserve critical facts?
- Did context management trigger too early/late?
- Were tools filtered incorrectly?

## Subagents
Ask:
- Was delegation necessary?
- Was the right specialist chosen?
- Was work duplicated?
- Did parent synthesis improve?

## Web Search
Ask:
- Did each search add new evidence?
- Were sources authoritative?
- Did the agent stop when evidence was sufficient?

# 14.15 — Feature-specific regression cases

Every important observed failure should become a regression case.

Examples:

```text
Add memory
→ test cross-thread personalization
→ test changing web facts are NOT remembered

Add Skills
→ test correct routing
→ test simple task avoids irrelevant skill

Add subagents
→ test complex delegation
→ test simple task stays with parent

Add summarization
→ test important fact survives compaction
```

This is how the eval suite becomes a durable engineering asset.

In [ ]:
REGRESSION_CASE_TEMPLATE = {
    "id": "feature_failure_name",
    "complexity": "medium",
    "prompt": "...",
    "expected": {"behavior": "..."},
    "protects_against": "description of observed failure",
}

# 14.16 — Change one meaningful variable at a time

Bad experiment:

```text
new prompt
+ new model
+ memory
+ Skills
+ subagents
```

If quality changes, you cannot explain why.

Prefer:

```text
baseline
→ one meaningful intervention
→ rerun same evals
```

This gives causal understanding.

# 14.17 — Avoid eval overfitting

Failure mode:

```text
optimize repeatedly against 10 prompts
   ↓
excellent benchmark score
   ↓
worse general research behavior
```

Protect against this with:
- DEV cases
- held-out cases
- refreshed cases
- qualitative trace inspection
- real production-like scenarios

Never let one aggregate number replace judgment.

# 14.18 — Reusable improvement loop

```text
OBSERVE
inspect traces

   ↓

CLASSIFY
failure taxonomy

   ↓

HYPOTHESIZE
what mechanism caused it?

   ↓

MEASURE
targeted eval

   ↓

CHANGE
smallest intervention

   ↓

RERUN
same dataset / judge / metrics

   ↓

DECIDE
keep / revert / investigate
```

This should become the default agent-development workflow.

In [ ]:
IMPROVEMENT_LOOP = [
    "observe",
    "classify",
    "hypothesize",
    "measure",
    "change_one_thing",
    "rerun",
    "keep_or_revert",
]

IMPROVEMENT_LOOP

# 14.19 — Compact experiment log

After many iterations, an experiment log becomes extremely valuable.

In [ ]:
EXPERIMENT_LOG_COLUMNS = [
    "timestamp",
    "change_id",
    "baseline",
    "candidate",
    "observed_failure",
    "hypothesis",
    "change",
    "quality_delta",
    "token_delta",
    "latency_delta",
    "search_delta",
    "decision",
    "notes",
]

experiment_log = pd.DataFrame(columns=EXPERIMENT_LOG_COLUMNS)
experiment_log

# 14.20 — Economics bridge

For every improvement, track:

```text
Δ Quality
Δ Task Success
Δ Tokens
Δ Latency
Δ Tool Calls
Δ Searches
Δ Subagent Calls
Δ Complexity
```

Then ask:

> Was the improvement worth the added work?

Tracing tells you **where work happened**.

Evaluation tells you **whether quality improved**.

Agent economics tells you **whether that improvement was worth it**.

## 14.20A — Avoid “quality gain per token” as the only metric

Suppose:

```text
Baseline quality = 4.1
Candidate quality = 4.6
Required threshold = 4.5
```

The candidate crosses the success threshold.

That may matter far more than a simple quality-per-token ratio.

Keep these separate first:

```text
quality
task success
tokens
latency
tool cost
```

Then connect them to outcome and value.

# 14.21 — Recommended cadence

### During development
Run a small DEV suite after meaningful changes.

### Before merge / release
Run DEV + held-out.

### Periodically
Add failures found in:
- traces
- customer scenarios
- production-like tests
- new capabilities

The suite should evolve with the agent.

# 14.22 — What belongs in production vs the notebook?

## Production package
Keep reusable mechanics:
- tracing
- trajectory extraction
- deterministic evaluators
- rubric helpers
- usage aggregation
- agent builders

## Notebook
Keep experimental orchestration:
- datasets
- variant definitions
- hypotheses
- comparisons
- plots
- experiment log
- human interpretation

The notebook is the **lab**, not the runtime.

# 14.23 — Minimum reusable workflow

After future changes, ideally edit only:

```text
1. candidate agent builder
2. new regression case, if needed
```

Then rerun:

```python
baseline_records, candidate_records = await compare_variants(...)
```

Everything else should remain stable.

That is what makes this a reusable improvement notebook rather than a one-time demo.

# Notebook 14 — Key takeaways

1. Trace = **what happened**.
2. Eval = **was it good**.
3. Final quality alone cannot diagnose agent behavior.
4. Use a stable failure taxonomy.
5. Turn trace symptoms into measurable hypotheses.
6. Keep the dataset compact and representative.
7. Run deterministic checks before expensive judging.
8. Evaluate trajectory as well as final answer.
9. Keep baseline/candidate plumbing identical.
10. Change one meaningful variable at a time.
11. Add regression cases for every important failure.
12. Separate DEV and held-out cases.
13. Avoid benchmark overfitting.
14. Track quality, success, tokens, latency, and work separately.
15. Let economics decide whether an improvement earns its extra cost.

Final loop:

```text
Trace
  ↓
Failure taxonomy
  ↓
Hypothesis
  ↓
Targeted eval
  ↓
Small change
  ↓
Rerun
  ↓
Quality + trajectory + economics
  ↓
Keep / revert
```